### author by yangshichen
### 注意：脚本仅供参考，使用前请仔细阅读

### 准备ma文件

#### hg19转hg38

In [ ]:
#!/bin/bash

set -e

input_dir="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/GWAS_ma/bbj-2/"
rename="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/chr_rename.txt"
chain="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/hg19ToHg38.over.chain.gz"
ref="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Homo_sapiens_assembly38.fasta.gz"

threads=20

process_vcf() {

vcf="$1"
base=$(basename "$vcf" .vcf.gz)

chr_vcf="${input_dir}/${base}.chr.vcf.gz"
hg38_vcf="${input_dir}/${base}.hg38.vcf.gz"
reject="${input_dir}/${base}.rejected.vcf"

echo "Processing $base"

# 1 rename chr
bcftools annotate \
--rename-chrs "$rename" \
"$vcf" \
-Oz -o "$chr_vcf"

# 2 index
bcftools index "$chr_vcf"

# 3 liftover
gatk LiftoverVcf \
-I "$chr_vcf" \
-O "$hg38_vcf" \
-CHAIN "$chain" \
-R "$ref" \
--REJECT "$reject" \
--RECOVER_SWAPPED_REF_ALT true

# 4 clean
rm -f "$chr_vcf" "${chr_vcf}.csi" "$reject"

echo "$base done"

}

export -f process_vcf
export input_dir rename chain ref

find "$input_dir" -maxdepth 1 -name "*.vcf.gz" \
| grep -v ".hg38" \
| while read vcf; do
    ((i=i%threads)); ((i++==0)) && wait
    process_vcf "$vcf" &
done

wait

echo "All finished."

In [ ]:
nohup bash liftover_parallel.sh > liftover.log 2>&1 &

#### 转ma

In [ ]:
#!/bin/bash

set -e

input_dir="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/GWAS_ma/bbj-2/"
output_dir="/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/GWAS_ma/ma_file_for_SMR_BBJ-2/"

for vcf in "$input_dir"/*.hg38.vcf.gz
do
    base=$(basename "$vcf" .vcf.gz)
    output="$output_dir/${base}.ma"

    echo "Processing $base"

    bcftools query \
    -f '%CHROM\t%POS\t%ALT\t%REF\t[%AF]\t[%ES]\t[%SE]\t[%LP]\n' \
    "$vcf" | \
    awk 'BEGIN{OFS="\t";print "SNP","A1","A2","freq","b","se","p"}
    $5!="." && $6!="." && $7!="." && $8!="." {p = 10^(-$8); snp = $1"_"$2; print snp,$3,$4,$5,$6,$7,p}' > "$output"

    echo "$base done"
done

### 准备epi/esi文件

In [1]:
import pandas as pd
import numpy as np
import os

In [ ]:
plink \
  --vcf /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Genetics/WGS_HP_imputed.variants.snp.filtered.vcf.gz \
  --make-bed \
  --out 10.maf01

In [2]:
eQTL_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL/'
eQTL_temp_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/result_for_besd/'
eQTL_output_dir = '/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/besd/'
smr = '/media/scPBMC1_AnalysisDisk1/huangzhuoli/Script_HPC/software_gaoyue/SMR/smr-1.3.1-linux-x86_64/smr-1.3.1'

In [3]:
CT_list = ['Adaptive NK cells','ALPL- MARCKS- NDNs','ASDC','Atypical naïve B cells','Basophils','CCR4- CD8+ Tcm','CCR4+ CD8+ Tcm','CD14+ cDC2','CD177int iLDNs','CD1C+ cDC2','CD27- IgD- atypical memory B cells','CD27- MAIT','CD27- Th1','CD27- Th17','CD27+ IgD- atypical memory B cells','CD27+ IgD+ atypical memory B cells','CD27+ MAIT','CD27+ Th1','CD27+ Th17','CD279+ SOX4+ Vδ1+ T cells','CD4+ Temra','CD56+ MAIT','CD56bright NK cells','CD56dim NK cells','CD62Lhi GZMK+ Vδ2+ T cells','CD8+ Temra','CD8+ Treg','CD95 memory B cells','cDC1','CILCP','CLP','Core classical monocytes','Core NDNs','CXCL8- PTGS2+ NDNs','DN T cells','Early memory B cells','FOS- NDNs','GBP1+ classical monocytes','GZMB+ CD4+ terminal effector T cells','GZMB+ CD8+ Tem','GZMB+ Vδ2+ T cells','GZMK+ CD8+ Tem','GZMK+ effector Vδ1+ T cells','GZMK+ Vδ2+ T cells','HLA-DRhi CD4+ terminal effector T cells','HLA-DRhi CD4+ Treg','HLA-DRhi CD8+ Tem','HSC_MPP','IFIT2- RNF213- NDNs','IGKChi Plasma cells','IGLL5hi Plasma cells','ILC2','ILCP','iNKT','Intermediate monocytes','IRF1- GBP2- NDNs','ISG+ classical monocytes','KLRB1+ CD4+ Treg','KLRC2+ effector Vδ1+ T cells','LAMP3+ DC','MC_MCP','Memory CD4+ Treg','MEP','MkP','MMP8+ CD177+ iLDNs','MMP8+ CD177+ mLDNs','MMP9+ CD177+ iLDNs','MPO- CD177- iLDNs','MPO+ CD177- iLDNs','MPO+ mLDNs','MT-ATP6- MT-CO2- NDNs','Naïve B cells','Naïve CD4+ T cells','Naïve CD4+ Treg','Naïve CD8+ T cells','Naïve Vδ1+ T cells','Non-classical monocytes','Non-switched memory B cells','pDCs','Plamsablasts','Platelets','Proliferative CD4+ Treg','Proliferative CD8+ memory T cells','Proliferative cytotoxic CD4+ T cells','Proliferative DN T cells','Proliferative help memory T cells','Proliferative iLDNs','Proliferative mLDNs','Proliferative NK cells','Proliferative Vδ2+ T cells','RGS2- NDNs','SOX4+ Vδ1+ T cells','Switched Memory B cells','Tfh','Th1_Th17','Th2','Th22',
           'VIM- FLNA- NDNs','vNKT']

In [4]:
#update_esi
bim = pd.read_csv('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/Genetics/10.maf01.bim', header=None, sep='\t')
bim.columns = ['chr','variant_id','dis','pos','A1','A2']
bed_df = pd.read_csv('/media/AnalysisDisk2/Yangshichen/0_HIV_RNA/QTL/01.Dynamic/01.Data/01.Genotype/gene_annotation.txt',sep='\t')
bed_df['chr'] = bed_df['chr'].str.replace('chr', '', regex=False)
bed_df = bed_df[bed_df['chr'].str.isdigit()]
bed_df['chr'] = bed_df['chr'].astype(int)

In [5]:
for celltype in CT_list:
    print('processing_' + celltype)

    # 读取显著 eGene
    eGene = pd.read_csv('/media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Result/eQTL_all_lead_perm_qvalues_0.05.csv')
    eGene = eGene[eGene['celltype'] == celltype]
    eGene = eGene['phenotype_id'].unique()
    
    # 读取该 celltype 下所有 parquet 文件
    parquet_files = [f for f in os.listdir(f"{eQTL_dir}{celltype}") if f.endswith('.parquet')]

    eQTL_match = pd.DataFrame()
    for file in parquet_files:
        file_path = os.path.join(f"{eQTL_dir}{celltype}", file)
        eQTL_result = pd.read_parquet(file_path)
        eQTL_result_match = eQTL_result[eQTL_result['phenotype_id'].isin(eGene)]
        eQTL_match = pd.concat([eQTL_match, eQTL_result_match])
    
    # make besd 文件
    eQTL_match_besd = eQTL_match.loc[:, ['variant_id','phenotype_id','slope','slope','pval_nominal','pval_nominal']]
    eQTL_match_besd.columns = ['SNP','gene','beta','t-stat','p-value','FDR']
    eQTL_match_besd['t-stat'] = 'NA'
    eQTL_match_besd['FDR'] = 'NA'

    eqtl_txt = f"{eQTL_temp_dir}/{celltype}_eQTL_for_besd.txt"
    eQTL_match_besd.to_csv(eqtl_txt, index=False, sep='\t')

    # Linux command：注意加引号
    gi = f'{smr} --eqtl-summary "{eqtl_txt}" --matrix-eqtl-format --make-besd --out "{eQTL_output_dir}{celltype}"'
    os.system(gi)
    
    print('updating_' + celltype)

    # 更新 .esi 文件
    esi_path = f"{eQTL_output_dir}{celltype}.esi"
    esi = pd.read_csv(esi_path, sep='\t', header=None)
    esi.columns = ['chr','variant_id','dis','pos','A1','A2','af']
    esi = esi[['variant_id']]
    esi = pd.merge(esi, bim.loc[:, ['chr','variant_id','dis','pos','A1','A2']], on='variant_id')

    af_df = eQTL_match.loc[:, ['variant_id','af']]
    af_df = af_df.loc[~af_df.duplicated(subset='variant_id', keep='first')]
    esi = pd.merge(esi, af_df, on='variant_id')
    esi = esi.loc[:, ['chr','variant_id','dis','pos','A1','A2','af']]

    esi_update_path = f"{eQTL_output_dir}{celltype}_update.esi"
    esi.to_csv(esi_update_path, sep='\t', header=None, index=False)

    gi = f'{smr} --beqtl-summary "{eQTL_output_dir}{celltype}" --update-esi "{esi_update_path}"'
    os.system(gi)

    # 更新 .epi 文件
    epi_path = f"{eQTL_output_dir}{celltype}.epi"
    epi = pd.read_csv(epi_path, sep='\t', header=None)
    epi.columns = ['chr','prob','dis','pos','gene_id','strand']
    epi[['gene_id']] = epi[['prob']]
    epi = epi[['prob','gene_id']]
    epi = pd.merge(epi, bed_df, on='gene_id')
    epi['dis'] = 0
    epi['strand'] = '+'
    epi = epi[['chr', 'prob', 'dis', 'left', 'gene_id', 'strand']]

    epi_update_path = f"{eQTL_output_dir}{celltype}_update.epi"
    epi.to_csv(epi_update_path, index=False, header=None, sep='\t')

    gi = f'{smr} --beqtl-summary "{eQTL_output_dir}{celltype}" --update-epi "{epi_update_path}"'
    os.system(gi)

processing_Adaptive NK cells
*******************************************************************
* Summary-data-based Mendelian Randomization (SMR)
* Version 1.3.1
* Build at Sep 21 2022 12:13:19, by GCC 8.3
* (C) 2015 Futao Zhang, Zhihong Zhu and Jian Yang
* The University of Queensland
* MIT License
*******************************************************************
Analysis started: 17:50:25,Fri Feb 27,2026

Options:
--eqtl-summary /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/result_for_besd//Adaptive NK cells_eQTL_for_besd.txt
--matrix-eqtl-format 
--make-besd 
--out /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/besd/Adaptive NK cells

Reading eQTL summary data from /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/result_for_besd//Adaptive NK cells_eQTL_for_besd.txt ...
16254478 rows to be included from /media/AnalysisDisk2/Yangshichen/2_My-Onek/WGS/Data/SMR_eQTL_besd/result_for_besd//Adaptive NK cells_eQTL_for_besd.txt.

Ge